In [10]:
import numpy as np
import math
import vtk

from array import array
from vtk.util import numpy_support
from tqdm import tqdm

In [11]:
def npArray2Vtk(data, fileName, spacing_x, spacing_y, spacing_z):
    # Convert the numpy array to a VTK array
    vtk_data_array = numpy_support.numpy_to_vtk(num_array=data.ravel(order='F'), deep=True, array_type=vtk.VTK_FLOAT)

    # Create a VTK image data object
    image_data = vtk.vtkImageData()

    # Set the dimensions of the image data (same as the shape of the numpy array)
    image_data.SetDimensions(data.shape)
    image_data.SetSpacing(spacing_x, spacing_y, spacing_z)
    
    #Set origin 
    x_origin = (data.shape[0] - 1)*spacing_x/2
    y_origin = (data.shape[1] - 1)*spacing_y/2
    z_origin = (data.shape[2] - 1)*spacing_z/2
    image_data.SetOrigin(-x_origin, -y_origin, -z_origin)

    # Allocate scalars for the image data
    image_data.AllocateScalars(vtk.VTK_FLOAT, 1)

    # Get the VTK array from the image data object and set its values to the converted numpy array
    vtk_array = image_data.GetPointData().GetScalars()
    vtk_array.DeepCopy(vtk_data_array)

    # Create a VTK writer to save the image data to a .vtk file
    writer = vtk.vtkStructuredPointsWriter()
    writer.SetFileName(fileName)
    writer.SetInputData(image_data)

    # Write the .vtk file
    writer.Write()

    print("3D volume data has been saved to", fileName)

    return image_data

## Read in Chameleon dataset
### Dimention: 1024 x 1024 x 1080

In [12]:
filePath = "data/chameleon_1024x1024x1080_uint16.raw"
f = open(filePath, "r")
data = np.fromfile(f, dtype=np.uint16)
print(data.shape)
data = np.reshape(data, (1024, 1024, 1080), order='F')
print(data.shape)

(1132462080,)
(1024, 1024, 1080)


In [13]:
# Down sample to half the size
data_ds = data[::2, ::2, ::2]
print("data_ds shape: ", data_ds.shape)
data_dds = data_ds[::2, ::2, ::2]
print("data_dds shape: ", data_dds.shape)
data_ddds = data_dds[::2, ::2, ::2]
print("data_ddds shape: ", data_ddds.shape)

data_ds shape:  (512, 512, 540)
data_dds shape:  (256, 256, 270)
data_ddds shape:  (128, 128, 135)


In [14]:
data_dds_crop = data_dds[:, :, 0:256]
print("data_dds_crop shape:", data_dds_crop.shape)

data_dds_crop shape: (256, 256, 256)


# after experiemnt 256 cube with 1 spaciang works the best for this dataset using 0.1 as sample distance

In [15]:
data = data_dds_crop
data = (data - np.min(data))/(np.max(data) - np.min(data))

In [16]:
data.dtype

dtype('float64')

In [17]:
spacing = 1
imageData = npArray2Vtk(data, "chameleon_256x256x256_normalized_space-1.vtk", spacing, spacing, spacing)

3D volume data has been saved to chameleon_256x256x256_normalized_space-1.vtk


## Save as ply

In [18]:
# save numpy volume data to .npy file

dims = imageData.GetDimensions()

reshaped_data = np.reshape(data, (dims[2], dims[1], dims[0]), order='F')
np.save("chameleon_256x256x256_normalized_space-1.npy", reshaped_data)